<a href="https://colab.research.google.com/github/shubham-bari/Language-Models/blob/main/MLA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch as torch
from torch import nn
import math

In [4]:
class MLA(nn.Module):

  def __init__(self, d_model, n_heads, kv_latent_dim):
    super().__init___()
    assert d_model % n_heads == 0, "d_in must be divisible by n_heads"

    self.d_model = d_model
    self.n_heads = n_heads
    self.kv_latent_dim = kv_latent_dim
    self.d_heads = d_model // n_heads

    #projection layers
    self.W_q = nn.Linear(d_model, d_model, bias=False)
    self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias = False)
    self.W_uk = nn.Linear(kv_latent_dim, d_model, bias = False)
    self.W_uv = nn.Linear(kv_latent_dim, d_model, bias = False)
    self.W_o = nn.Linear(d_model, d_model, bias=False)

    self.ln = nn.LayerNorm(kv_latent_dim)
    self.register('absorbed_k', None)

  def forward(self, x, kv_cache=None, past_len=0):
    B, S, D=x.size()

    if self.absorbed_k is None:
      absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight) #shape = (d_model, latent_dim)
      absorbed_k = absorbed.view(self.n_heads, self.d_heads, -1)   # shape = (n_heads, d_heads, latent_dim)

    #compress kv into latent space
    self.new_C_kv = self.ln(self.W_dkv(x))   #shape (B,S,latent_dim)

    if self.kv_cache is None:
      C_kv = self.new_C_kv
    else:
      C_kv = torch.cat([self.kv_cache, self.new_C_kv], dim=1)   #dim=1 means concatenating along seq_len

    S_new = C_kv.size(1)

    #decompress V and split into heads
    v_full = self.W_uv(C_kv)  # shape = (B, S_new, d_model)
    v = v_full.view(B, S_new, self.n_heads, self.d_heads).transpose(1,2)

    q = x.view(B, S, self.n_heads, self.d_heads).transpose(1,2)

    #compute attn scores
    attn_scores = torch.zeroes(B, self.n_heads, S, S_new)
    for h in range(self.n_heads):
      tmp = torch.matmul(q[:,:,h], self.absorbed_k[h])    #(B,S,latent_dim)
      attn_scores[:,h] = torch.bmm(tmp, C_kv.transpose(1,2))   #(B,S,S_new)

    #scaling+masking
    attn_scores = attn_scores / math.sqrt(self.d_heads)
    mask = torch.tril(torch.ones((S, S_new)), diagonal = past_len)
    attn_scores = attn_scores.masked_fill(mask.view(1,1,S,S_new)==0, float('-inf'))

    #softmax
    attn_weights = torch.F.softmax(attn_scores, dim=-1)

    out_heads=0
    for h in range(self.n_heads):
      context_h = torch.bmm(attn_weights[:,h], v[:,h])   #shape(B,S,d_heads)
      out_heads.append(context_h)

    out = torch.cat(out_heads, dim=-1)    #(B,S,D)

    return self.W_o(out), C_kv



